# Submission Ensemble CV

## Scores:
### anchor_override_m2h3_l2m3.csv: .97800
### majority_tie_anchor.csv: .97787
### weighted_lb.csv: .97787
### weighted_lb_highboost_0.08.csv: .97787
### weighted_lb_highboost_0.12.csv: .97787
### weighted_lb_highboost_0.16.csv: .97787

In [1]:
from pathlib import Path
import re
import itertools
import numpy as np
import pandas as pd

ROOT = Path('.')
TARGET = 'Irrigation_Need'
ID_COL = 'id'

SCORE_FLOOR = 97700


def parse_score(fname):
    m = re.match(r'^score(\d+)\.csv$', fname)
    if not m:
        return None
    return int(m.group(1))


all_score_files = sorted([p.name for p in ROOT.glob('score*.csv')])
SUB_FILES = [f for f in all_score_files if (parse_score(f) is not None and parse_score(f) >= SCORE_FLOOR)]

if not SUB_FILES:
    raise ValueError(f'No score files found at or above {SCORE_FLOOR}.')

LB_WEIGHTS = {Path(f).stem: parse_score(f) / 100000.0 for f in SUB_FILES}

LABELS = ['Low', 'Medium', 'High']

OUT_DIR = ROOT / 'ensemble_outputs'
OUT_DIR.mkdir(exist_ok=True)

print('Using submissions:', SUB_FILES)
print('Parsed LB weights:', LB_WEIGHTS)

Using submissions: ['score97760.csv', 'score97770.csv', 'score97772.csv', 'score97793.csv', 'score97800.csv']
Parsed LB weights: {'score97760': 0.9776, 'score97770': 0.9777, 'score97772': 0.97772, 'score97793': 0.97793, 'score97800': 0.978}


In [2]:
def load_submissions(files):
    dfs = []
    names = []
    for f in files:
        path = ROOT / f
        n = Path(f).stem
        d = pd.read_csv(path)[[ID_COL, TARGET]].rename(columns={TARGET: n})
        dfs.append(d)
        names.append(n)
    merged = dfs[0]
    for d in dfs[1:]:
        merged = merged.merge(d, on=ID_COL, how='inner')
    return merged, names


def vote_row(row_vals, names, schema):
    if schema['kind'] == 'majority':
        counts = {k: 0.0 for k in LABELS}
        for n, v in zip(names, row_vals):
            counts[v] += 1.0
        mx = max(counts.values())
        cands = [k for k, v in counts.items() if v == mx]
        if len(cands) == 1:
            return cands[0]
        return row_vals[schema['tie_break_idx']]

    if schema['kind'] == 'weighted':
        scores = {k: 0.0 for k in LABELS}
        for n, v in zip(names, row_vals):
            scores[v] += schema['weights'][n]
        if schema.get('high_boost', 0.0) > 0:
            if abs(scores['High'] - scores['Medium']) <= schema.get('high_boost_margin', 0.2):
                scores['High'] += schema['high_boost']
        return max(scores, key=scores.get)

    if schema['kind'] == 'anchor_override':
        anchor = row_vals[schema['anchor_idx']]
        high_votes = int(np.sum(np.array(row_vals) == 'High'))
        med_votes = int(np.sum(np.array(row_vals) == 'Medium'))
        low_votes = int(np.sum(np.array(row_vals) == 'Low'))

        pred = anchor
        if anchor == 'Medium' and high_votes >= schema.get('med_to_high_votes', 3):
            pred = 'High'
        elif anchor == 'Low' and med_votes >= schema.get('low_to_med_votes', 4):
            pred = 'Medium'
        elif anchor == 'Medium' and low_votes >= schema.get('med_to_low_votes', 4):
            pred = 'Low'
        return pred

    raise ValueError(f"Unknown schema kind: {schema['kind']}")


def apply_schema(df, names, schema):
    vals = df[names].to_numpy()
    preds = [vote_row(row, names, schema) for row in vals]
    return np.array(preds, dtype=object)


def lomo_consistency_cv(df, names, schema, lb_weights):
    vals = df[names].to_numpy()
    schema_preds = apply_schema(df, names, schema)

    per_model = []
    for i, n in enumerate(names):
        held_out = vals[:, i]
        match = (schema_preds == held_out).mean()
        per_model.append((n, match, lb_weights.get(n, 0.0)))

    num = sum(m * w for _, m, w in per_model)
    den = sum(w for _, _, w in per_model)
    weighted_score = num / den if den > 0 else np.mean([m for _, m, _ in per_model])
    return weighted_score, per_model


merged, names = load_submissions(SUB_FILES)
print('Merged shape:', merged.shape)
merged.head()

Merged shape: (270000, 6)


,id,score97760,score97770,score97772,score97793,score97800
0,630000,Low,Low,Low,Low,Low
1,630001,Low,Low,Low,Low,Low
2,630002,Low,Low,Low,Low,Low
3,630003,Low,Low,Low,Low,Low
4,630004,Low,Low,Low,Low,Low


In [3]:
schemas = []

anchor_name = max(names, key=lambda n: LB_WEIGHTS.get(n, 0.0))
anchor_idx = names.index(anchor_name)
print('Anchor model:', anchor_name, '| LB weight:', LB_WEIGHTS.get(anchor_name, 0.0))

schemas.append({
    'name': 'majority_tie_anchor',
    'kind': 'majority',
    'tie_break_idx': anchor_idx,
})

w_base = {n: LB_WEIGHTS.get(n, 1.0) for n in names}
w_norm = sum(w_base.values())
w_base = {k: v / w_norm for k, v in w_base.items()}

schemas.append({
    'name': 'weighted_lb',
    'kind': 'weighted',
    'weights': w_base,
    'high_boost': 0.0,
    'high_boost_margin': 0.2,
})

for hb in [0.08, 0.12, 0.16]:
    schemas.append({
        'name': f'weighted_lb_highboost_{hb:.2f}',
        'kind': 'weighted',
        'weights': w_base,
        'high_boost': hb,
        'high_boost_margin': 0.25,
    })

for m2h, l2m in [(3, 4), (2, 4), (3, 3), (2, 3)]:
    schemas.append({
        'name': f'anchor_override_m2h{m2h}_l2m{l2m}',
        'kind': 'anchor_override',
        'anchor_idx': anchor_idx,
        'med_to_high_votes': m2h,
        'low_to_med_votes': l2m,
        'med_to_low_votes': 4,
    })

rows = []
for sc in schemas:
    cv_score, per_model = lomo_consistency_cv(merged, names, sc, LB_WEIGHTS)
    preds = apply_schema(merged, names, sc)
    changes_vs_anchor = int((preds != merged[names[anchor_idx]].to_numpy()).sum())
    high_count = int((preds == 'High').sum())
    rows.append({
        'schema': sc['name'],
        'kind': sc['kind'],
        'lomo_score': cv_score,
        'changes_vs_anchor': changes_vs_anchor,
        'high_count': high_count,
    })

rank_df = pd.DataFrame(rows).sort_values(['lomo_score', 'changes_vs_anchor'], ascending=[False, True]).reset_index(drop=True)
rank_df

Anchor model: score97800 | LB weight: 0.978


,schema,kind,lomo_score,changes_vs_anchor,high_count
0,majority_tie_anchor,majority,0.999219,254,10575
1,weighted_lb,weighted,0.999219,254,10575
2,weighted_lb_highboost_0.08,weighted,0.999219,254,10575
3,weighted_lb_highboost_0.12,weighted,0.999219,254,10575
4,weighted_lb_highboost_0.16,weighted,0.999219,254,10575
5,anchor_override_m2h3_l2m3,anchor_override,0.999180,207,10584
6,anchor_override_m2h3_l2m4,anchor_override,0.999179,206,10584
7,anchor_override_m2h2_l2m3,anchor_override,0.999099,317,10694
8,anchor_override_m2h2_l2m4,anchor_override,0.999098,316,10694


In [4]:
TOP_K = 6
anchor_name = names[anchor_idx]

selected = rank_df.head(TOP_K).copy()
print(selected)

for _, r in selected.iterrows():
    sname = r['schema']
    schema = next(x for x in schemas if x['name'] == sname)
    pred = apply_schema(merged, names, schema)
    out = pd.DataFrame({ID_COL: merged[ID_COL], TARGET: pred})
    out_path = OUT_DIR / f'{sname}.csv'
    out.to_csv(out_path, index=False)
    print('saved', out_path)

print('Done.')

                       schema             kind  lomo_score  changes_vs_anchor  \
0         majority_tie_anchor         majority    0.999219                254   
1                 weighted_lb         weighted    0.999219                254   
2  weighted_lb_highboost_0.08         weighted    0.999219                254   
3  weighted_lb_highboost_0.12         weighted    0.999219                254   
4  weighted_lb_highboost_0.16         weighted    0.999219                254   
5   anchor_override_m2h3_l2m3  anchor_override    0.999180                207   

   high_count  
0       10575  
1       10575  
2       10575  
3       10575  
4       10575  
5       10584  
saved ensemble_outputs\majority_tie_anchor.csv
saved ensemble_outputs\weighted_lb.csv
saved ensemble_outputs\weighted_lb_highboost_0.08.csv
saved ensemble_outputs\weighted_lb_highboost_0.12.csv
saved ensemble_outputs\weighted_lb_highboost_0.16.csv
saved ensemble_outputs\anchor_override_m2h3_l2m3.csv
Done.
